## Sentiment Analysis

### Classify documents based on their sentiment using ML

We will download the movie review dataset from IMDB.

In [36]:
import tarfile
import pandas as pd
import numpy as np
import pyprind
import re
import os
import sys
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer

In [ ]:
# Unzip tar file 
with tarfile.open("data/aclImdb_v1.tar","r") as tar:
    tar.extractall(path="./data/")

In [10]:
basepath = 'data/aclImdb'

In [ ]:
## Reading reviews from both test and train dataset and appending them into one dataset and creating a label.
labels = {'pos': 1, 'neg': 0}
pbar = pyprind.ProgBar(50000, stream = sys.stdout)
df = pd.DataFrame()

for s in ('test', 'train'):
    for l in ('pos','neg'):
        path = os.path.join(basepath, s, l)
        for file in sorted(os.listdir(path)):
            with open(os.path.join(path, file), 'r', encoding = 'utf-8') as infile:
                txt = infile.read()
            df = df.append([[txt, labels[l]]],
                           ignore_index = True)
            pbar.update()

df.columns = ['review', 'sentiment']

0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:04:05


In [ ]:
## Reshuffle the index so we have a mix of both positive and negative review.
np.random.seed(0)
df = df.reindex(np.random.permutation(df.index))
# save the data.
df.to_csv('data/movie_data.csv', index = False, encoding='utf-8')

In [23]:
df = pd.read_csv('data/movie_data.csv', encoding='utf-8')
df.head()

,review,sentiment
0,"In 1974, the teenager Martha Moxley (Maggie Gr...",1
1,OK... so... I really like Kris Kristofferson a...,0
2,"***SPOILER*** Do not read this, if you think a...",0
3,hi for all the people who have seen this wonde...,1
4,"I recently bought the DVD, forgetting just how...",0


## Bag-of-words model

Bag of word model allow us to represent text as numerical features vectors.

The idea behind bag-of-words model is :-
1. we create a vocabulary of unique tokens - for example, words - from the entire set of documents.
2. we construct  a feature vector from each document that contains the count of how often each word occurs in the particular document.

Since the unique word in a document is a small subset of bag-of-words vocabulary, most feature vector will be 0, hence we call them sparse.

### Transform words into feature vectors

We can use CountVectorizer that take an array of text and create bag-of-words model for us.

In [ ]:
## An Example
count = CountVectorizer()
docs = np.array(['The sun is shining',
        'The weather is sweet',
        'The sun is shining, the weather is sweet, and one and one is two'])
bag = count.fit_transform(docs) # construct vocab of bag-of-world model and transformed them into feature vector

In [ ]:
# bag-of-words vocabulary
count.vocabulary_

{'the': 6,
 'sun': 4,
 'is': 1,
 'shining': 3,
 'weather': 8,
 'sweet': 5,
 'and': 0,
 'one': 2,
 'two': 7}

The values for each word is their index. which means that 'and' has index 0, 'is' has index 1 and so on. The indexes are usually assigned alphabetically. So feature vector the value in index 0 will represent how many occurance of word 'and' has in a particulat text.

In [ ]:
## Sparse feature vectors
print(bag.toarray())

[[0 1 0 1 1 0 1 0 0]
 [0 1 0 0 0 1 1 0 1]
 [2 3 2 1 1 1 2 1 1]]


The values in the feature vectors are also called ***raw term frequencies***. 

***tf(t,d)*** - number of times a word t occur in the document d.

#### N-grams model

The contigous sequence of items in NLP - words, letters or symbols are also n-grams. The choice of number n depends on the kind of application. For e.g. in spam filtering, n-grams of size 3 or 4 yield good results. 

For a document "The sun is shining", 1-gram, 2-gram representation would be:-
* 1-gram: "The", "sun", "is", "shining".
* 2-gram: "The sun", "sun is", "is shining".


## term frequency-inverse document frequency (tf-idf)

During analysis, we often come across words that appear in all the documents from all classes. These frequently used words don't contain any valuable of differentiating information. We can use tf-idf to downweight such words in the feature vectors. The tf-idf can be defined as :-

$$tf-idf(t,d) = tf(t,d) * idf(t,d)$$
where idf(t,d) is given as 

$$idf(t,d) = log \frac{n_d}{1 + df(d,t)}$$
where $n_d$ is the total number of documents and df(t,d) is no. of documents contain the word t. The log is used to ensure low frequency words are not given too much weight. 1 is added just to make sure we have non-zero value of idf.


In [32]:
tfidf = TfidfTransformer(use_idf= True,
                         norm='l2',
                         smooth_idf=True)
np.set_printoptions(precision=2)
print(tfidf.fit_transform(count.fit_transform(docs)).toarray())


[[0.   0.43 0.   0.56 0.56 0.   0.43 0.   0.  ]
 [0.   0.43 0.   0.   0.   0.56 0.43 0.   0.56]
 [0.5  0.45 0.5  0.19 0.19 0.19 0.3  0.25 0.19]]


We can see that word "is" that has highest frequency in 3rd document is downweighted and now has tf-idf of 0.45 as it is also contained in other documents hence unlikely to have any useful information.

However, the scikit-learn has a different formula for idf.
$$idf(t,d) = log \frac{1 + n_d}{1 + df(d,t)}$$
and tf-idf would be 
$$tf-idf(t,d) = tf(t,d) * (idf(t,d) + 1)$$
The +1 is for setting smooth-idf = True, which is helpful for assigning 0 weight to terms occur in all documents.

It is also common to normalize term frequencies before calculating tf-idf. However, sklearn directly normalize tf-idfs. For L2 norm,
$$v_{norm} = \frac{v}{||v||_2}$$

## Cleaning text data

Text data may contain a lot of punctuation, HTML tags, other non letter characters. Some punctuations can be useful in some context.

In [34]:
df.loc[0,'review'][-50:]

'is seven.<br /><br />Title (Brazil): Not Available'

To remove these html, punctuation tags, we can use regex library. Word capitalization can be useful in some context but here we will convert everything to lower case.

In [37]:
def preprocessor(text):
    # For removing html/xml tags
    # map < bracket and Matches zero or more characters of any kind, except for a closing angle bracket using [^>]*
    # followed by a closing bracket >. which will remove html tags like <div>, etc.
    text = re.sub('<[^>]*>','',text) 

    # For removing emoticons by identifying eyes, nose, and mouth
    # (?::|;|=) - maps eyes as :, ;, =
    # (?:-) - maps nose as hyphen - exactly zero or one time because of the ? quantifier.
    # (?:\)|\(|D|P) - maps mouth as \, D, P
    # he ?: inside the parentheses creates non-capturing groups. 
    # This ensures re.findall returns the full matched emoticon string rather than just pieces of it.
    emoticons = re.findall('(?::|;|=)(?:-)?(?:\)|\(|D|P)', text)

    # For removing all spaces, punctuation, symbols
    # [\W]+ - [] is for character class. \W means all non-word items.
    # + - Matches one or more of these non-word characters in a row to speed up substitution
    text = (re.sub('[\W]+', '', text.lower()) + 
            ' '.join(emoticons).replace('-', '')) # removing nose emoticons for consistency
    
    return text
